# Mini Project 1 — Analysis Notebook

**Your name: Ning Zhang** 

**Dataset:**

FY24 Historical Reservations Full.csv (Recreation.gov historical reservations export); this notebook uses the processed subset `data/processed/yellowstone_summer_2024.csv` (Yellowstone National Park, `inventorytype` = CAMPING, stay start dates in June–August 2024). Optional context: RIDB facility metadata from recreation.gov RIDB API when cited.

**Date:**


In [2]:
# Setup — run this cell first
# If any package is missing, it will install automatically
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["pandas", "plotly", "kaleido", "nbformat"]:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        install(pkg)

import pandas as pd
import plotly.express as px

print("Setup complete.")

Setup complete.


---

## Section 1 — Overview

**Dataset:** Two federal datasets from the Recreation One Stop system. (1) RIDB API (`ridb.recreation.gov/api/v1`) — facility-level metadata for Yellowstone campgrounds including amenities, campsite attributes, and site types, accessed via REST API with a free API key. (2) Recreation.gov Historical Reservation CSV (`ridb.recreation.gov/download`) — filtered to Yellowstone National Park camping reservations (`parentlocation == "Yellowstone National Park"`, `inventorytype == "CAMPING"`) for summer 2024 (June–August), joined to the RIDB data on shared `facilityid`.

**Why this dataset:** Public land reservation systems are a high-stakes digital access interface, and understanding how visitors navigate booking at America's most iconic park connects directly to HCD questions about discoverability, equity, and whether design is concentrating demand or distributing it.

**Three analytical questions:**

1. Which Yellowstone campgrounds have the highest cancellation rates in summer 2024, and do those same facilities also show the shortest average booking lead times — suggesting frustrated demand rather than genuine disinterest?
2. Do campgrounds with more listed amenities (per RIDB API) generate higher reservation volume or longer average stays, or are simpler campgrounds equally in demand?
3. Which campgrounds are being reserved the furthest in advance in summer 2024, and what does that pattern suggest about perceived desirability across Yellowstone's facilities?

**What a practitioner would do with these findings:** A national park service UX or operations team could use these findings to redesign how campground information is surfaced in the reservation interface — prioritizing underbooked but comparable facilities to reduce overcrowding and improve access equity.

---
## Section 2 — Data Profile

Read the markdown, then run each code cell under **2.1**–**2.4**. Output is **text only** (no charts).


### 2.1 How many rows and columns?

**Reservations:** one row per historical booking; fixed width (35 fields). **RIDB:** row counts depend on which endpoints you stack (facilities under Yellowstone rec area 2988, then campsites per facility).


In [ ]:
from pathlib import Path
import json
import os
import ssl
import urllib.request

import pandas as pd

csv_path = Path("data/processed/yellowstone_summer_2024.csv")
df = pd.read_csv(csv_path, low_memory=False)
n_rows, n_cols = df.shape
print("Reservations CSV:", n_rows, "rows,", n_cols, "columns")

def _load_ridb_key():
    for fname in (".env", "yellowstone.env"):
        path = Path(fname)
        if not path.is_file():
            continue
        for line in path.read_text(encoding="utf-8").splitlines():
            s = line.strip()
            if s.startswith("RIDB_API_KEY="):
                return s.split("=", 1)[1].strip().strip('"').strip("'")
    return os.environ.get("RIDB_API_KEY", "").strip()

def _ridb_facility_count():
    key = _load_ridb_key()
    if not key:
        return None
    url = "https://ridb.recreation.gov/api/v1/recareas/2988/facilities?limit=500"
    req = urllib.request.Request(
        url,
        headers={"accept": "application/json", "apikey": key},
        method="GET",
    )
    try:
        import certifi
        ctx = ssl.create_default_context(cafile=certifi.where())
    except ImportError:
        ctx = ssl.create_default_context()
    with urllib.request.urlopen(req, context=ctx, timeout=60) as resp:
        data = json.loads(resp.read().decode("utf-8"))
    return len(data.get("RECDATA") or [])

ridb_n = _ridb_facility_count()
if ridb_n is None:
    print("RIDB: could not fetch (missing RIDB_API_KEY in .env / yellowstone.env or network error).")
else:
    print("RIDB facilities (rec area 2988, limit=500):", ridb_n, "records")


Reservations CSV: 15326 rows, 35 columns
RIDB facilities (rec area 2988, limit=500): 25 records


### 2.2 What does each column represent?

**Reservations:** mix of ids, hierarchy, facility/geo, timing, party/equipment, and payment fields (see `df.info()`). **RIDB:** nested JSON (addresses, media, campsites, attributes) keyed by `FacilityID`/`FacilityName`.


In [7]:
from pathlib import Path
import pandas as pd

df = pd.read_csv(Path("data/processed/yellowstone_summer_2024.csv"), low_memory=False, nrows=8000)
print("Column dtypes (8k-row sample):")
print(df.dtypes.value_counts())
print("\nColumn names:")
print(list(df.columns))


Column dtypes (8k-row sample):
str        18
float64    11
int64       6
Name: count, dtype: int64

Column names:
['historicalreservationid', 'ordernumber', 'agency', 'orgid', 'codehierarchy', 'regioncode', 'regiondescription', 'parentlocationid', 'parentlocation', 'legacyfacilityid', 'park', 'sitetype', 'usetype', 'productid', 'inventorytype', 'facilityid', 'facilityzip', 'facilitystate', 'facilitylongitude', 'facilitylatitude', 'customerzip', 'tax', 'usefee', 'tranfee', 'attrfee', 'totalbeforetax', 'discount', 'totalpaid', 'startdate', 'enddate', 'orderdate', 'nights', 'numberofpeople', 'equipmentdescription', 'equipmentlength']


### 2.3 Data quality snapshot

**Reservations:** sparse optional fields, `nights` as text, heterogeneous datetimes. **RIDB:** nested lists need flattening; rare non-numeric facility ids in other extracts.


In [ ]:
from pathlib import Path
import pandas as pd

df = pd.read_csv(Path("data/processed/yellowstone_summer_2024.csv"), low_memory=False)
miss = df.isna().mean().sort_values(ascending=False).head(15)
print("Top missingness (share of rows):")
print(miss.to_string())

nights_head = df["nights"].astype(str).str.extract(r"(\d+)", expand=False)
nd = pd.to_numeric(nights_head, errors="coerce")
print("\nLeading digit from `nights` text (value_counts, top 10):")
print(nd.dropna().astype(int).value_counts().head(10).to_string())


Top missingness (share of rows):
usefee                  1.000000
tranfee                 1.000000
attrfee                 1.000000
customerzip             0.187655
equipmentdescription    0.017226
sitetype                0.000065
usetype                 0.000065
equipmentlength         0.000065
enddate                 0.000065
numberofpeople          0.000065
totalpaid               0.000000
facilitylatitude        0.000000
totalbeforetax          0.000000
startdate               0.000000
orderdate               0.000000

Leading digit from `nights` text (value_counts, top 10):
nights
1     9268
2     2962
3     1568
4      715
5      338
6      145
7      129
8       54
14      44
9       35


### 2.4 Analysis focus

**Reservations:** `facilityid`, `orderdate`/`startdate`, `park`, `nights`, `numberofpeople`, `totalpaid`. **RIDB:** campsite/attribute payloads keyed by `FacilityID` for amenity depth and maps.


In [ ]:
from pathlib import Path
import pandas as pd

df = pd.read_csv(Path("data/processed/yellowstone_summer_2024.csv"), low_memory=False)
g = (
    df.groupby("park", observed=True)
    .agg(
        reservations=("historicalreservationid", "count"),
        median_paid=("totalpaid", "median"),
        mean_paid=("totalpaid", "mean"),
    )
    .sort_values("reservations", ascending=False)
)
print("Reservations by campground (focus columns preview):")
print(g.to_string())


In [26]:
# Check column types and missing values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id                 500 non-null    int64
 1   app                500 non-null    str  
 2   category           500 non-null    str  
 3   rating             500 non-null    int64
 4   review             500 non-null    str  
 5   date               500 non-null    str  
 6   helpful_votes      500 non-null    int64
 7   verified_purchase  500 non-null    bool 
 8   device_type        437 non-null    str  
 9   app_version        389 non-null    str  
dtypes: bool(1), int64(3), str(6)
memory usage: 35.8 KB


In [27]:
# Summary statistics for numeric columns
df.describe()

,id,rating,helpful_votes
count,500.000000,500.000000,500.000000
mean,250.500000,3.946000,23.464000
std,144.481833,1.184013,13.766471
min,1.000000,1.000000,0.000000
25%,125.750000,3.000000,11.000000
50%,250.500000,4.000000,23.500000
75%,375.250000,5.000000,35.000000
max,500.000000,5.000000,47.000000


**Your data profile notes:**  
*(Replace this with your observations — what's in the data, what you noticed, what questions it raises.)*

---

## Section 3 — Analysis

Answer your three research questions using pandas. Each question should have:

1. A markdown cell stating the question
2. A code cell with the analysis
3. A markdown cell with your interpretation — what does the result mean?

You may need to clean or reshape the data before you can answer a question. That's normal — document what you did and why.

**Question 1:** *(paste your first research question from MP1a here)*

In [28]:
# Your analysis for Question 1


**Interpretation:**  
*(What does this result tell you? Is it what you expected? What would you want to investigate further?)*

**Question 2:** *(paste your second research question here)*

In [29]:
# Your analysis for Question 2


**Interpretation:**  
*(What does this result tell you?)*

**Question 3:** *(paste your third research question here)*

In [30]:
# Your analysis for Question 3


**Interpretation:**  
*(What does this result tell you?)*

---

## Section 4 — Visualization

Create at least one visualization that supports one of your analysis findings. Your chart should:

- Have a title that states the finding, not just the data (e.g., "Satisfaction scores drop sharply after age 40" not "Satisfaction by age")
- Have labeled axes
- Use a chart type appropriate for your data (bar for categorical comparison, scatter for relationships, line for trends over time)

Below the chart, explain in a markdown cell: why you chose this chart type, and what you want the reader to take away from it.

In [31]:
# Your visualization
# Example structure — replace with your actual columns and finding

# fig = px.bar(
#     df,
#     x="your_category_column",
#     y="your_value_column",
#     title="Your finding stated as a claim",
#     labels={"your_category_column": "Readable label", "your_value_column": "Readable label"}
# )
# fig.show()


**Chart rationale:**  
*(Why this chart type? What should the reader take away?)*

---

## Section 5 — Conclusions

Write 3–5 sentences summarizing what you found. Address these questions:

- What is the most important thing your analysis revealed?
- What surprised you?
- What would you investigate next if you had more time or data?
- What are the limitations of this analysis — what can't you conclude from this data?

Then complete the competency claim below.

**Summary of findings:**  
*(Write your 3–5 sentence conclusion here.)*

---

## Competency Claim

In a `mp1.md` file in your GitHub repository, write a short competency claim (2–4 sentences) for each domain you feel this project demonstrates. Be specific — cite something you actually did in this notebook.

Domains covered by this project typically include:
- **C3 — Data cleaning and file handling** (if you cleaned or reshaped data)
- **C5 — Data analysis with pandas** (answering questions with code)
- **C6 — Data visualization** (your chart)
- **C7 — Critical evaluation and professional judgment** (your interpretation and limitations section)

You don't have to claim every domain — only the ones your work actually demonstrates.